In [2]:
from data_cleanup import functions
regions = functions.regions

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

In [3]:
#angepasst auf finale datei
seq_regions = ['SEQ_H1', 'SEQ_H2', 'SEQ_L1', 'SEQ_L2', 'SEQ_L3']
cf_regions = ['CF_H1', 'CF_H2', 'CF_L1', 'CF_L2', 'CF_L3']
df = (
    pd.read_csv("data/ab_ag_scalop.tsv", sep="\t")
    .dropna(subset=seq_regions)
    .dropna(subset=cf_regions)
    .drop_duplicates(subset=seq_regions) 
)
antigen_labels = df["antigen_name"].tolist()

In [7]:
feature_spaces = {}

seq_regions = ["SEQ_H1", "SEQ_H2", "SEQ_L1", "SEQ_L2", "SEQ_L3"]
cf_regions = ["CF_H1", "CF_H2", "CF_L1", "CF_L2", "CF_L3"]

df = (
    pd.read_csv("data/ab_ag_scalop.tsv", sep="\t")
    .dropna(subset = seq_regions)
    .dropna(subset = cf_regions)
    .drop_duplicates(subset = seq_regions)
)

df_rows = df[["pdb", "Hchain", "Lchain", "model", "antigen_name", "antigen_species"]].values.tolist()

for seq_region in seq_regions:
    fs_rows = []
    for df_row in df_rows:
        pdb = df_row[0]
        hchain = df_row[1]
        lchain = df_row[2]
        model = df_row[3]
        antigen_name = df_row[4]
        antigen_species = df_row[5]

        if seq_region[4] == "H": #z.B. "H" in "SEQ_H1" oder "L" in "SEQ_L2"
            chain_id = hchain
        else:
            chain_id = lchain

        # Embedding-Koordinaten der PDB-Einträge
        embedding_as_arr = np.load(f"data/embeddings/{pdb}/{pdb}_{chain_id}_{seq_region[4]}_{seq_region}_chothia.npy")
        #embedding_as_vec = embedding_as_arr.mean(axis=0)
        embedding_as_vec = PCA(n_components=1).fit_transform(embedding_as_arr.T).flatten()

        # Erstellen und Anhängen der Zeile eines PDB-Eintrages für den Feature-Space
        fs_row = [pdb, hchain, lchain, model, antigen_name, antigen_species] + embedding_as_vec.tolist()
        fs_rows.append(fs_row)

    # Umwandeln des Feature-Space von einer List in einen DataFrame
    feature_space = pd.DataFrame(fs_rows, columns = ["pdb", "Hchain", "Lchain", "model", "antigen_name", "antigen_species"] + [f"embedding_dimension {e}" for e in range(embedding_as_vec.shape[0])])
    feature_spaces[seq_region] = feature_space

In [ ]:
#hierarchisches Clustering der ESMC-Embeddings
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram

# -------------- Konfiguration --------------
# Pfad zum Ordner mit den ESMC-Embeddings
emb_dir = "./data/esmc_embeddings_scalop"  # passe den Pfad an dein Repo an
# Dateiendung der Embedding-Dateien
emb_ext = ".npy"

# OPTIONAL: DataFrame mit Feature-Space (falls Du bereits einen hast)
# Lege hier Deinen DataFrame an oder lade ihn, z.B.:
# feature_space = pd.read_csv("dein_feature_space.tsv", sep='\t', index_col=0)
feature_space = None

# OPTIONAL: Liste von Labels (Index im DataFrame oder Dateinamen ohne Endung),
# die Du clustern möchtest. Wenn leer oder None, werden alle verwendet.
filter_labels = []  # z.B. ["1ABC_H_SEQ_H1", "2XYZ_L_SEQ_L3"]

# -------------- Feature-Space aus DataFrame nutzen --------------
if feature_space is not None:
    # Spalten mit Embedding-Dimensionen erkennen (prefix emb_dim_)
    emb_cols = [c for c in feature_space.columns if c.startswith("emb_dim_")]
    if not emb_cols:
        raise ValueError("Keine Spalten mit 'emb_dim_' im feature_space DataFrame gefunden.")
    X = feature_space[emb_cols].values
    labels = feature_space.index.astype(str).tolist()
    # Wenn filter_labels gesetzt, filtern
    if filter_labels:
        mask = [lbl in filter_labels for lbl in labels]
        X = X[mask]
        labels = [lbl for lbl, keep in zip(labels, mask) if keep]
    print(f"Verwendete Embeddings (DataFrame): {X.shape[0]} Objekte, {X.shape[1]} Dimensionen")
else:
    # -------------- Alle .npy-Dateien rekursiv finden --------------
    emb_files = []
    for root, dirs, files in os.walk(emb_dir):
        for f in files:
            if f.endswith(emb_ext):
                emb_files.append(os.path.join(root, f))
    emb_files.sort()
    if not emb_files:
        raise ValueError(
            f"Keine '{emb_ext}'-Dateien im Verzeichnis '{emb_dir}' (inkl. Unterverzeichnisse) gefunden."
        )
    # -------------- Embeddings laden und optional mitteln --------------
    emb_list, labels = [], []
    for filepath in emb_files:
        label = os.path.splitext(os.path.basename(filepath))[0]
        if filter_labels and label not in filter_labels:
            continue
        emb = np.load(filepath)
        if emb.ndim > 1:
            emb = emb.mean(axis=0)
        emb_list.append(emb)
        labels.append(label)
    if not emb_list:
        raise ValueError(
            f"Nach Filterung keine Embeddings übrig. Filter-Liste: {filter_labels}"
        )
    try:
        X = np.vstack(emb_list)
    except Exception:
        shapes = [arr.shape for arr in emb_list]
        raise ValueError(
            "Fehler beim Stapeln: unterschiedliche Dimensionen gefunden.\n"
            f"Shapes: {shapes}"
        )
    print(f"Verwendete Embeddings (Dateien): {X.shape[0]} Objekte, {X.shape[1]} Dimensionen")

# -------------- Hierarchisches Clustering --------------
Z = linkage(X, method="ward", metric="euclidean")

# -------------- Dendrogramm plotten --------------
plt.figure(figsize=(10, 6))
dendrogram(
    Z,
    labels=labels,
    leaf_rotation=90,
    leaf_font_size=8,
    color_threshold=None
)
plt.title("Hierarchisches Clustering der ESMC-Embeddings")
plt.xlabel("Objekte")
plt.ylabel("Distanz (Ward)")
plt.tight_layout()
plt.show()


ValueError: Keine '.npy'-Dateien im Verzeichnis './data/esmc_embeddings_scalop' (inkl. Unterverzeichnisse) gefunden.

In [ ]:
data_cleanup.functions.show_PCA(feature_spaces, 3)

In [ ]:
data_cleanup.functions.elbow(feature_spaces, 3)

In [ ]:
data_cleanup.functions.silhouette(feature_spaces, 3)

In [ ]:
# Auswahl der optimalen Clusterzahlen für silhouette_check und show_Kmeans
cluster_counts = {
    "CDR_H1": 4,
    "CDR_H2": 6,
    "CDR_L1": 5,
    "CDR_L2": 7,
    "CDR_L3": 2
}

In [ ]:
data_cleanup.functions.silhouette_check(feature_spaces, cluster_counts, ncols=3)

In [ ]:
data_cleanup.functions.show_Kmeans(feature_spaces, cluster_counts, 3)

In [ ]:
#Proportionstest => testen, ob die Antigen-Namen systematisch unterschiedlich auf die Cluster verteilt sind

#nullhypothese (H0): Verteilung der Antigen-Namen auf die Cluster ist zufällig
#H1: Verteilung der Antigen-Namen auf die Cluster ist nicht zufällig

for region in regions:

    # Kontingenztabelle (Antigen x Cluster) erzeugen

    # automatische Zählung und erstellen von dataframe über crosstab
    contingency_table = pd.crosstab(pd.Series(antigen_labels, name='Antigen'), pd.Series(cluster_labels, name='Cluster'))

    #kontingenztabelle printen
    print(f"\n Kontingenztabelle für {region}")
    print(contingency_table)

    #chi-quadrat-test
    chi2, p, dof, expected = chi2_contingency(contingency_table)
    #chi2=maß für wie stark die beobachteten häufigkeiten von den erwarteten abweichen (je größer, desto größer abweichung)
    #p-Wert=Wahrscheinlichkeit, bei zufälliger Verteilung eine Abweichung mindestens so groß wie chi2 zu beobachten
    #dof=degrees of freedom (#zeilen-1 * #spalten-1)
    #expected=tabelle mit erwarteten häufigkeiten unter nullhypothese

    #print ergebnisse
    print(f"\n Chi-Quadrat-Test Ergebnis für {region}")
    print(f"Chi2-Wert: {chi2:.4f}") #4f für 4 nachkommSTELLEN
    print(f"p-Wert:    {p:.4e}")  # 4e für exponentielle darstellung mit 4 nachkommastellen
    print(f"Freiheitsgrade: {dof}")

    #print die expected werte zum vergleich
    expected_df = pd.DataFrame(expected, index=contingency_table.index, columns=contingency_table.columns)
    print(f"\nErwartete Häufigkeiten (unter Nullhypothese):")
    print(expected_df.round(2)) #auf 2 nachkommastellen runden

    #die tatsache dass die antigen_name häufigkeiten sehr unterschiedlich sind wird hier automatisch berücksichtigt: 
    #expected-Array wird aus den Randhäufigkeiten berechnet => randhäufigkeiten für antigen =  Summe pro Zeile => wie oft kommt jedes Antigen insgesamt im ganzen Datensatz vor?
    #randhäufigkeiten für cluster=Summe pro Spalte => wie groß ist jedes Cluster insgesamt?

    #wenn p < 0.05 => signifikant also die verteilung unterschiedet sich signifikant von zufall => N0 hypothese muss verworfen werden denn die unterschiede in der verteilung sind nicht durch zufall zu erklären


In [ ]:
# Brauchen wir erst mal nicht!
'''# Alternative: Darstellung der Embeddings über Hausdorff-Distanz und Multidimensional Scaling
from scipy.spatial.distance import directed_hausdorff
from sklearn.manifold import MDS

subset_df = df.sample(300, random_state=42)


for region in regions:
    alle_embeddings = []
    labels = []
    for i, df_row in subset_df.iterrows():
        pdb = df_row["pdb"]
        antigen_name = df_row["antigen_name"]
        #antigen_species = df_row["antigen_species"]
        embedding_as_arr = np.load(f"data/embeddings/{pdb}/{pdb}_{region}.npy")
        alle_embeddings.append(embedding_as_arr)
        labels.append(antigen_name)
    n = len(alle_embeddings)
    dist_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            dist = directed_hausdorff(alle_embeddings[i], alle_embeddings[j])[0]
            dist_matrix[i, j] = dist
            dist_matrix[j, i] = dist
    mds = MDS(n_components=2, dissimilarity='precomputed', random_state=42)
    coords = mds.fit_transform(dist_matrix)

    label_codes = pd.Categorical(labels).codes
    label_categories = pd.Categorical(labels).categories
    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(coords[:, 0], coords[:, 1], c=label_codes, cmap='tab10', s=20)
    plt.legend(handles=scatter.legend_elements()[0], labels=list(label_categories), title="Antigen", loc="best")
    plt.title(f"Embeddings der {region} (Hausdorff)")
    plt.grid(True)
    plt.show()'''